# Construction Permits – Uster

**Pipeline:**
1. Load cadastral plots (GeoPackage → GeoJSON)
2. Fetch raw permit publications from amtsblattportal.ch
3. Parse addresses from titles (handles multiple addresses per permit)
4. Explode to one row per address (same `id`)
5. Geocode: swisstopo API (primary) → Nominatim (fallback)
6. Spatial join with plots to assign parcel polygon geometry
7. Export `construction_permits.geojson`
8. Fetch public transport stops (Overpass / OpenStreetMap)
9. **Part 3**: Interactive map – plots with permits + plots near transit

> **Data sources:** amtsblattportal.ch (permits), swisstopo (geocoding + cadastre), OpenStreetMap via Overpass API (transit stops)

> **Coordinate reference system:** All three output files (`plots.geojson`, `construction_permits.geojson`, `stops.geojson`) use **WGS84 (EPSG:4326)** — the standard for web maps and GeoJSON. Where metric accuracy is needed (e.g. the 250 m transit buffer in Part 3), data is temporarily reprojected to Swiss LV95 (EPSG:2056) and then back to WGS84.

In [ ]:
# ── Install dependencies  ────────────────────────────────────────────
# !pip install geopandas polars lxml geopy folium tqdm pyogrio

# ── Configuration ──────────────────────────────────────────────────────────────────
# Adjust DATA_DIR to your local path or Google Drive mount point
# On Colab: from google.colab import drive; drive.mount('/content/drive')
#           DATA_DIR = "/content/drive/MyDrive/PopetyTest/"

from pathlib import Path
DATA_DIR  = Path("/Users/nathalieguibert/Downloads/PopetyTest/")
GPKG_PATH = DATA_DIR / "79cb1aec092f4642bcc0cdcdbeb499c2/AV_MOpublic-_Liegenschaften_-OGD/AV_MOpublic-_Liegenschaften_-OGD.gpkg"

In [ ]:
import re, time, json
from io import BytesIO
from pathlib import Path

import requests
import lxml.etree as LET
import polars as pl
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point
from tqdm import tqdm
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderQuotaExceeded

## 0. Cadastral plots (Uster)

**Data source:** Canton Zurich open data — `AV_MOpublic` GeoPackage (amtliche Vermessung).  
Filtered to municipality 198 (Uster) using the `bfsnr` attribute.

### Attributes kept

The raw GeoPackage contains many technical survey fields. For real-estate purposes, the following were selected:

| Column | Description | Why relevant |
|---|---|---|
| `egris_egrid` | Federal parcel identifier (EGRID) | Unique, stable ID — links to cadastral register, GWR, and other federal datasets |
| `nummer` | Local parcel number | Used for display and cross-referencing with municipal records |
| `flaechenmass` | Parcel area in m² | Core real-estate attribute — directly affects land value and development potential |
| `geometry` | Parcel polygon | Required for all spatial operations |

All other columns (survey metadata, cantonal codes, administrative references) were dropped as they are not directly relevant to real-estate analysis.

### CRS

The source data uses **Swiss LV95 (EPSG:2056)**. We export two versions:
- `plots_lv95.geojson` — original metric CRS, used for distance calculations
- `plots_wgs84.geojson` / `plots.geojson` — reprojected to **WGS84 (EPSG:4326)**, the unified CRS for all three pipeline outputs

In [ ]:
# Load only the Uster parcels (bfsnr = 198) from the cantonal GeoPackage
liegenschaften_f = gpd.read_file(
    str(GPKG_PATH),
    layer="avzh_liegenschaften_f",
    where="bfsnr=198",
)
print(f"Loaded {len(liegenschaften_f)} parcels, CRS: {liegenschaften_f.crs}")
print(f"Columns: {list(liegenschaften_f.columns)}")

# Export in Swiss LV95 (metric, for buffer calculations)
liegenschaften_f.to_file(str(DATA_DIR / "plots_lv95.geojson"), driver="GeoJSON")

# Export in WGS84 (for geocoding boundary validation and Folium maps)
plots_wgs84 = liegenschaften_f.to_crs(epsg=4326)
plots_wgs84.to_file(str(DATA_DIR / "plots_wgs84.geojson"), driver="GeoJSON")

# Save a slim version as plots.geojson (used by the Airflow DAG)
plots_slim = plots_wgs84[["egris_egrid", "nummer", "flaechenmass", "geometry"]]
plots_slim.to_file(str(DATA_DIR / "plots.geojson"), driver="GeoJSON")

print(f"Exported plots_lv95.geojson, plots_wgs84.geojson, plots.geojson")
liegenschaften_f.plot(figsize=(8, 8), color="lightblue", edgecolor="gray")

In [ ]:
# Visual verification: plots should cover the Uster municipality area
plots_wgs84.explore(
    color="lightblue",
    tooltip=["egris_egrid", "nummer", "flaechenmass"],
    name="Plots (Uster)",
)

## 1. Fetch raw permits from amtsblattportal.ch

The Swiss Official Gazette API (`amtsblattportal.ch`) provides building permit publications in XML format.
We filter by sub-rubric `BP-ZH01` (Zurich canton building permits) and municipality 198 (Uster).

In [ ]:
url = "https://amtsblattportal.ch/api/v1/publications/xml"
params = {
    "publicationStates": "PUBLISHED",
    "subRubrics":        "BP-ZH01",
    "municipalityId":    "198",
    "pageRequest.page":  0,
    "pageRequest.size":  200,
}
r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
raw_xml = r.text
print(f"Status: {r.status_code}  |  Size: {len(raw_xml)} chars")

## 2. Parse XML → Polars DataFrame

Use `lxml.iterparse` for memory-efficient streaming XML parsing.
Each `<publication>` element is read and immediately cleared from memory.

In [ ]:
def parse_permits_xml(xml_string: str) -> pl.DataFrame:
    """Stream-parse the XML response into a Polars DataFrame.
    
    Uses lxml iterparse to avoid loading the full XML tree into memory.
    Each <publication> element is cleared after reading.
    """
    data = []
    context = LET.iterparse(
        BytesIO(xml_string.encode("utf-8")),
        events=("end",),
        tag="{*}publication",
    )
    for _, elem in context:
        meta = elem.find(".//{*}meta")
        reg  = meta.find(".//{*}registrationOffice") if meta is not None else None
        data.append({
            "id":          meta.findtext("{*}id")               if meta else None,
            "pub_date":    meta.findtext("{*}publicationDate")   if meta else None,
            "title":       meta.findtext("{*}title/{*}de")       if meta else None,
            "office_name": reg.findtext("{*}displayName")        if reg  else None,
        })
        elem.clear()
        while elem.getprevious() is not None:
            del elem.getparent()[0]

    return pl.DataFrame(data).with_columns(
        pl.col("pub_date").str.to_date("%Y-%m-%d")
    )

permits_raw = parse_permits_xml(raw_xml)
print(f"Fetched {permits_raw.shape[0]} permits")
permits_raw.head(5)

## 3. Address parsing

Extract street addresses from free-text permit titles like:
> *"Bauprojekt: Guldenenstrasse 14-18, Assek. Nrn. 123/456, Uster"*

### Rules
- Strip leading `"bei "` (*near*, address is still usable if a number follows)
- Remove everything from `"Assek."` onwards (building-registry annotations)
- Split on `/` **only** when the right side starts with a street-name keyword → two separate streets
- Split on `,` **only** when the next token looks like a new street name
- Expand number ranges: `"14-18"` → `["14", "16", "18"]`
- Drop address parts with **no digit** (field names, area designations) — cannot be geocoded

In [ ]:
# Regex to detect Swiss-German street-type suffixes
_STREET_RE = re.compile(
    r"(strasse|str\.?|weg|gasse|platz|allee|rain|steig|stieg|halde"
    r"|holz|riet|berg|bühl|bach|wies(?:en)?|roos|matt(?:en)?"
    r"|feld|graben|ring|bogen|park|gässli|ufer|dorf)",
    re.IGNORECASE,
)

def _looks_like_street(s: str) -> bool:
    return bool(_STREET_RE.search(s))


def _expand_number_range(street: str, raw_range: str) -> list[str]:
    """Expand a house-number range into individual addresses.

    Examples:
      "Guldenenstrasse", "14-18"      → ["Guldenenstrasse 14", "Guldenenstrasse 16", "Guldenenstrasse 18"]
      "Krämerackerstrasse", "11/11a-11d" → ["Krämerackerstrasse 11", "Krämerackerstrasse 11a", ...]
    """
    # Letter range: "11/11a-11d"
    m = re.match(r'^(\d+)\s*/\s*\d+([a-z])-\d+([a-z])$', raw_range, re.IGNORECASE)
    if m:
        base = int(m.group(1))
        results = [f"{street} {base}"]
        for c in range(ord(m.group(2).lower()), ord(m.group(3).lower()) + 1):
            results.append(f"{street} {base}{chr(c)}")
        return results

    # Numeric range: "14-18"
    m = re.match(r'^(\d+)-(\d+)$', raw_range)
    if m:
        start, end = int(m.group(1)), int(m.group(2))
        step = 2 if (end - start) % 2 == 0 and end > start + 1 else 1
        return [f"{street} {n}" for n in range(start, end + 1, step)]

    return [f"{street} {raw_range}"]


def _expand_same_street_numbers(street: str, numbers: list[str]) -> list[str]:
    return [f"{street} {n}" for n in numbers if n.strip()]

In [ ]:
def parse_addresses(title: str) -> list[str]:
    """Extract geocodable addresses from a permit title.
    
    Returns a list of full addresses like "Hauptstrasse 14, Uster, 8610, Switzerland".
    Returns [] when no geocodable address can be found.
    """
    if not title:
        return []

    # Extract free-text after "Bauprojekt:"
    m = re.search(r"Bauprojekt:\s*(.+)", title)
    if not m:
        return []
    raw = m.group(1)

    # Remove building-registry annotations (everything from "Assek." onwards)
    raw = re.split(r",?\s*[Aa]ssek\.?", raw)[0].strip()

    # Normalise: "bei " → remove prefix; "bei Nr. X" → "X"; " und " → "/"
    raw = re.sub(r"^bei\s+", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s+bei\s+Nr\.?\s*", " ", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s+und\s+(?=\d)", "/", raw, flags=re.IGNORECASE)

    # Step 1: split on "/" — only when right side looks like a new street
    slash_parts = re.split(r"\s*/\s*", raw)
    merged: list[str] = []
    current = slash_parts[0]
    for part in slash_parts[1:]:
        if _looks_like_street(part):
            digit_prefix = re.match(r'^(\d[\w]*)\s*,\s*(.+)', part)
            if digit_prefix:
                current = current + "/" + digit_prefix.group(1)
                merged.append(current.strip())
                current = digit_prefix.group(2).strip()
            else:
                merged.append(current.strip())
                current = part.strip()
        else:
            current = current + "/" + part
    merged.append(current.strip())

    # Step 2: split on "," — only when next token is a new street
    street_parts: list[str] = []
    for part in merged:
        sub = re.split(r",\s*", part)
        cur = sub[0]
        for s in sub[1:]:
            if _looks_like_street(s):
                street_parts.append(cur.strip())
                cur = s.strip()
            else:
                cur = cur + ", " + s
        street_parts.append(cur.strip())

    # Step 3: expand ranges and filter
    valid: list[str] = []
    for addr in street_parts:
        addr = addr.strip()
        if not addr:
            continue
        m_street = re.match(r'^(.+?)\s+(\d[\w/\-]*)$', addr)
        if m_street:
            street_name = m_street.group(1).strip()
            num_part    = m_street.group(2).strip()
            if re.match(r'^\d+-\d+$', num_part):
                valid.extend(f"{a}, Uster, 8610, Switzerland" for a in _expand_number_range(street_name, num_part))
                continue
            if re.match(r'^\d+/\d+[a-z]-\d+[a-z]$', num_part, re.IGNORECASE):
                valid.extend(f"{a}, Uster, 8610, Switzerland" for a in _expand_number_range(street_name, num_part))
                continue
            slash_nums = num_part.split("/")
            if len(slash_nums) > 1 and all(re.match(r'^\d+[a-z]?$', p, re.IGNORECASE) for p in slash_nums):
                valid.extend(f"{a}, Uster, 8610, Switzerland" for a in _expand_same_street_numbers(street_name, slash_nums))
                continue
        if re.search(r"\d", addr) or _looks_like_street(addr):
            valid.append(f"{addr}, Uster, 8610, Switzerland")
    return valid

In [ ]:
# ── Unit tests for parse_addresses ───────────────────────────────────────────────
test_cases = [
    ("Bauprojekt: Chammerholzstr. 10/Fehraltorferstrasse 18a/18b, Assek. Nrn. 5709/5573, Uster",
     ["Chammerholzstr. 10, Uster, 8610, Switzerland",
      "Fehraltorferstrasse 18a/18b, Uster, 8610, Switzerland"]),
    ("Bauprojekt: Sandstrasse 2, Seestrasse 26/28, Assek. Nr. 2591, 2578, Uster",
     ["Sandstrasse 2, Uster, 8610, Switzerland",
      "Seestrasse 26/28, Uster, 8610, Switzerland"]),
    ("Bauprojekt: Winterthurerstrasse 49/49a, Assek. Nr. 880/676, Uster",
     ["Winterthurerstrasse 49/49a, Uster, 8610, Switzerland"]),
    ("Bauprojekt: bei Schwerzistrasse 60, Uster",
     ["Schwerzistrasse 60, Uster, 8610, Switzerland"]),
    ("Bauprojekt: Hinterbreiti Kieswerk, Uster",
     []),
    ("Bauprojekt: Alpenblickstrasse 24/26, Asylstrasse 25/27, Wagerenstrasse 21/23, Uster",
     ["Alpenblickstrasse 24/26, Uster, 8610, Switzerland",
      "Asylstrasse 25/27, Uster, 8610, Switzerland",
      "Wagerenstrasse 21/23, Uster, 8610, Switzerland"]),
]

all_ok = True
for title, expected in test_cases:
    got = parse_addresses(title)
    ok  = got == expected
    if not ok:
        all_ok = False
    print(f"{'\u2713' if ok else '\u2717'}  {title[:70]}")
    if not ok:
        print(f"   expected: {expected}")
        print(f"   got:      {got}")

print("\nAll tests passed! \u2713" if all_ok else "\nSome tests FAILED")

## 4. Explode: one row per address

Permit titles can contain multiple addresses (e.g. a project spanning several parcels).
We explode each permit into one row per address, all sharing the same `id`.

In [ ]:
rows = []
for row in permits_raw.iter_rows(named=True):
    addresses = parse_addresses(row["title"])
    if not addresses:
        rows.append({**row, "project_address": None, "parse_status": "no_address"})
    else:
        for addr in addresses:
            status = "street_only" if not re.search(r"\d", addr) else "ok"
            rows.append({**row, "project_address": addr, "parse_status": status})

permits_exploded = pd.DataFrame(rows)

multi = len(permits_exploded) - len(permits_raw)
print(f"Original permits  : {len(permits_raw)}")
print(f"Rows after explode: {len(permits_exploded)}  (+{multi} from multi-address permits)")
print(permits_exploded["parse_status"].value_counts())
permits_exploded[["id", "title", "project_address", "parse_status"]].head(10)

## 5. Geocoding

### Strategy

| Step | Geocoder | Reason |
|------|----------|---------|
| Primary | **swisstopo** (`api3.geo.admin.ch`) | Swiss federal address register; covers every official Swiss address; no strict rate limit |
| Fallback 1 | **Nominatim** (OpenStreetMap) | Global coverage; useful for named buildings; limited to 1 req/s; only accepts `type=house/building` |
| Fallback 2 | Strip letter suffix | `"17a"` → `"17"` – try without suffix |
| Fallback 3 | Lower house number | Address not yet registered; try `15`, `13`, … |
| Fallback 4 | Street name only | Last resort; marks result as `needs_review:street_only` |

All results are validated against the Uster municipality boundary.
Results outside Uster are rejected (prevents swisstopo returning a similarly-named street elsewhere).

In [ ]:
# Load plots for boundary validation
plots = gpd.read_file(str(DATA_DIR / "plots_wgs84.geojson")).to_crs(epsg=4326)
uster_boundary = plots.dissolve().geometry.iloc[0]
print(f"Plots loaded: {len(plots)} parcels")

nominatim = Nominatim(user_agent="popety_permits_pipeline")


def _in_uster(lat: float, lon: float) -> bool:
    """Check if coordinates fall within the Uster municipality boundary."""
    return uster_boundary.contains(Point(lon, lat))


def geocode_swisstopo(addr: str) -> tuple[float, float] | None:
    """Query the swisstopo location API. Returns (lat, lon) or None.
    
    Validates that the returned street name matches the query to prevent
    false positives (e.g. a same-numbered street in another municipality).
    """
    addr = re.sub(r'(\d+)\.\d+', r'\1', addr)  # "87.2" → "87"
    params = {"type": "locations", "searchText": addr, "lang": "de", "limit": 1, "sr": "4326"}
    try:
        resp = requests.get("https://api3.geo.admin.ch/rest/services/api/SearchServer",
                            params=params, timeout=10)
        results = resp.json().get("results", [])
        if not results:
            return None
        attrs = results[0].get("attrs", {})
        street_part  = addr.split(",")[0]
        street_words = " ".join(w for w in street_part.split()
                                if not re.match(r'^\d', w) and len(w) > 2).lower()
        detail = attrs.get("detail", "").lower()
        if street_words and street_words not in detail:
            return None
        lat, lon = attrs.get("lat"), attrs.get("lon")
        if lat is not None and lon is not None:
            return float(lat), float(lon)
    except Exception:
        pass
    return None


def geocode_nominatim(addr: str) -> tuple[float, float] | None:
    """Nominatim fallback with retry on rate-limit.
    
    Only accepts results with type "house" or "building" — rejects interpolated
    street-midpoint results which Nominatim returns when the exact number is missing.
    """
    for attempt in range(3):
        try:
            loc = nominatim.geocode(addr, timeout=10, country_codes="ch")
            if loc:
                if loc.raw.get("type") not in ("house", "building"):
                    return None
                return loc.latitude, loc.longitude
            return None
        except GeocoderQuotaExceeded:
            time.sleep(30)
        except Exception as e:
            if "429" in str(e):
                time.sleep(5 * (attempt + 1))
            else:
                return None
    return None


def _try_geocode(addr: str) -> tuple[float, float] | None:
    """Try swisstopo then Nominatim for a single address."""
    c = geocode_swisstopo(addr)
    if c and _in_uster(*c):
        return c
    time.sleep(1.5)
    c = geocode_nominatim(addr)
    return c if c and _in_uster(*c) else None


def _strip_letter_suffix(num: str) -> str | None:
    """'17a' → '17',  '26f' → '26',  '14' → None"""
    m = re.match(r'^(\d+)[a-z]+$', num, re.IGNORECASE)
    return m.group(1) if m else None


def _decrement_number(street: str, num: str, max_steps: int = 5):
    """Try progressively lower house numbers (same odd/even parity).
    
    Used when a new building's address isn't yet registered in swisstopo.
    Returns (coords, tried_address) or (None, None).
    """
    base_m = re.match(r'^(\d+)', num)
    if not base_m:
        return None, None
    base = int(base_m.group(1))
    step = 2 if base > 1 else 1
    for i in range(1, max_steps + 1):
        candidate = base - i * step
        if candidate < 1:
            break
        c = _try_geocode(f"{street} {candidate}, Uster, 8610, Switzerland")
        if c:
            return c, f"{street} {candidate}"
    return None, None


def geocode_with_fallback(addr: str) -> tuple[float | None, float | None, str]:
    """Full geocoding pipeline with cascading fallbacks. Returns (lat, lon, status).
    
    Status values:
      "swisstopo"                  – exact match via swisstopo
      "nominatim"                  – exact match via Nominatim  
      "needs_review:no_suffix"     – letter suffix stripped (17a → 17)
      "needs_review:lower_number"  – used a lower house number
      "needs_review:street_only"   – only street name matched
      "failed"                     – no match found
    """
    addr = re.sub(r'(\d+)\.\d+', r'\1', addr)  # normalise "87.2" → "87"

    # 1. Exact match
    c = geocode_swisstopo(addr)
    if c and _in_uster(*c):
        return c[0], c[1], "swisstopo"
    time.sleep(1.5)
    c = geocode_nominatim(addr)
    if c and _in_uster(*c):
        return c[0], c[1], "nominatim"

    # Parse street + number
    san = addr.split(",")[0].strip()
    m   = re.match(r'^(.+?)\s+(\S+)$', san)
    if not m:
        return None, None, "failed"
    street, num = m.group(1), m.group(2)

    # 2. Strip letter suffix
    base_num = _strip_letter_suffix(num)
    if base_num:
        c = _try_geocode(f"{street} {base_num}, Uster, 8610, Switzerland")
        if c:
            return c[0], c[1], "needs_review:no_suffix"

    # 3. Try lower house numbers
    c, _ = _decrement_number(street, base_num or num)
    if c:
        return c[0], c[1], "needs_review:lower_number"

    # 4. Street name only
    c = geocode_swisstopo(f"{street}, Uster, 8610, Switzerland")
    if c and _in_uster(*c):
        return c[0], c[1], "needs_review:street_only"

    return None, None, "failed"

In [ ]:
# ── Geocoding cache: avoids re-geocoding known addresses ───────────────────────
# Cached results persist across runs in geocode_cache.json
CACHE_FILE = DATA_DIR / "geocode_cache.json"
_geocache: dict = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
print(f"Cache loaded: {len(_geocache)} known addresses")

def _save_cache():
    CACHE_FILE.write_text(json.dumps(_geocache, ensure_ascii=False, indent=2))

# ── Run geocoding ───────────────────────────────────────────────────────────────
to_geocode = permits_exploded["parse_status"] != "no_address"
print(f"Geocoding {to_geocode.sum()} addresses  ({(~to_geocode).sum()} skipped)...")

lats, lons, statuses = [], [], []
new_entries = 0

for _, row in tqdm(permits_exploded.iterrows(), total=len(permits_exploded), desc="Geocoding"):
    if row["parse_status"] == "no_address":
        lats.append(None); lons.append(None); statuses.append("skipped")
        continue
    addr = row["project_address"]
    if addr in _geocache:
        cached = _geocache[addr]
        lats.append(cached["lat"]); lons.append(cached["lon"]); statuses.append(cached["status"])
    else:
        lat, lon, status = geocode_with_fallback(addr)
        _geocache[addr] = {"lat": lat, "lon": lon, "status": status}
        new_entries += 1
        if new_entries % 10 == 0:
            _save_cache()
        lats.append(lat); lons.append(lon); statuses.append(status)

_save_cache()
permits_exploded["lat"] = lats
permits_exploded["lon"] = lons
permits_exploded["geocode_status"] = statuses

found = permits_exploded["lat"].notna().sum()
total = len(permits_exploded)
print(f"\nGeocoded: {found}/{total} ({found/total*100:.1f}%) — {new_entries} new entries added to cache")
print(permits_exploded["geocode_status"].value_counts(dropna=False))

## 6. Spatial join: assign parcel polygon

For each geocoded permit point, find the cadastral parcel it falls within.
The permit's final geometry **is the parcel polygon** — which is what Part 3 needs
(*"Plots of land that have an active construction permit on them"*).

Permits that could not be geocoded retain `geometry = null`.

In [ ]:
geocoded_mask = permits_exploded["lat"].notna()

# Build GeoDataFrame with point geometry for geocoded permits
geocoded_gdf = gpd.GeoDataFrame(
    permits_exploded[geocoded_mask].copy(),
    geometry=gpd.points_from_xy(
        permits_exploded.loc[geocoded_mask, "lon"],
        permits_exploded.loc[geocoded_mask, "lat"],
    ),
    crs="EPSG:4326",
)

# Spatial join: find which parcel each permit point falls within
plots_slim = plots[["egris_egrid", "nummer", "flaechenmass", "geometry"]].copy()
joined = gpd.sjoin(geocoded_gdf, plots_slim, how="left", predicate="within")

# Replace point geometry with matched parcel polygon
matched_mask = joined["index_right"].notna()
joined.loc[matched_mask, "geometry"] = (
    plots_slim.loc[joined.loc[matched_mask, "index_right"].astype(int), "geometry"].values
)

print(f"Geocoded permits:          {len(geocoded_gdf)}")
print(f"Matched to parcel polygon: {matched_mask.sum()}")
print(f"Point only (no parcel):    {(~matched_mask).sum()}")

# Combine geocoded (with parcel polygon) + non-geocoded (geometry=None)
not_geocoded = gpd.GeoDataFrame(
    permits_exploded[~geocoded_mask].copy(),
    geometry=[None] * (~geocoded_mask).sum(),
    crs="EPSG:4326",
)
keep_cols    = [c for c in joined.columns if c != "index_right"]
permits_final = gpd.GeoDataFrame(
    pd.concat([joined[keep_cols], not_geocoded.reindex(columns=keep_cols)], ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)
print(f"\nFinal rows:      {len(permits_final)}")
print(f"With geometry:   {permits_final.geometry.notna().sum()}")
print(f"Without geometry:{permits_final.geometry.isna().sum()}")

## 7. Document permits without geometry

In [ ]:
no_geom = permits_final[permits_final.geometry.isna()][
    ["id", "title", "project_address", "geocode_status"]
]
print(f"{len(no_geom)} permits without geometry:\n")
for _, row in no_geom.iterrows():
    reason = "no address extracted" if not row["project_address"] else "geocoding failed"
    print(f"  [{reason}]  {row['title'][:80]}")

## 8. Visual check – geocoding quality map

- 🟢 **Green** = exact match (swisstopo or Nominatim)
- 🟠 **Orange** = needs review (suffix stripped / lower number / street only)

Failed permits are excluded (no geometry).

In [ ]:
permits_with_geom = permits_final[permits_final.geometry.notna()].copy()

def _status_color(status: str) -> str:
    if status in ("swisstopo", "nominatim"):
        return "green"
    if isinstance(status, str) and status.startswith("needs_review"):
        return "orange"
    return "gray"

permits_with_geom["_color"] = permits_with_geom["geocode_status"].apply(_status_color)

m = plots.explore(color="lightgray", name="Parcels")
for color, group in permits_with_geom.groupby("_color"):
    label = {"green": "Exact match", "orange": "Needs review"}.get(color, color)
    group.explore(m=m, color=color,
                  tooltip=["title", "pub_date", "project_address", "geocode_status"],
                  name=label)
m.save(str(DATA_DIR / "permits_map.html"))
print("Saved permits_map.html")
m

## 9. Export

In [ ]:
permits_final["pub_date"] = permits_final["pub_date"].astype(str)

# Save permits without geometry to CSV for documentation
permits_no_geom = permits_final[permits_final.geometry.isna()].drop(columns="geometry")
permits_no_geom.to_csv(str(DATA_DIR / "permits_no_geometry.csv"), index=False)
print(f"Saved {len(permits_no_geom)} ungeocodable permits to permits_no_geometry.csv")

# Save geocoded permits to GeoJSON
permits_geojson = permits_final[permits_final.geometry.notna()].copy()
permits_geojson.to_file(str(DATA_DIR / "construction_permits.geojson"), driver="GeoJSON")
print(f"Saved {len(permits_geojson)} permits to construction_permits.geojson")

## 10. Public transport stops (Overpass API)

**Data source:** OpenStreetMap via the Overpass API.  
Queried using the Uster administrative boundary (`admin_level=8`).

Covers all node types relevant to public transport:
- `public_transport=stop_position` — exact boarding point
- `public_transport=platform` — waiting area
- `highway=bus_stop` — bus stops
- `railway=tram_stop`, `railway=station`, `railway=halt` — rail stops

### Attributes kept

| Column | Description | Why relevant |
|---|---|---|
| `osm_id` | OpenStreetMap node ID | Unique identifier, allows future updates/diffs |
| `name` | Stop name | Human-readable label for map display |
| `type` | Stop type (stop_position / platform / bus_stop / station) | Distinguishes transport modes |
| `network` | Transport network (e.g. ZVV) | Useful for filtering by operator network |
| `operator` | Operating company | Ownership context |
| `ref` | Line or stop reference number | Cross-reference with timetable data |
| `geometry` | Point geometry | Required for spatial operations (250 m buffer) |

Unnamed nodes (OSM noise, unmapped platforms) and exact duplicates (same name + location appearing as both stop_position and platform) are removed.

Uses POST requests and tries multiple Overpass mirror servers to avoid rate-limiting (the main `overpass-api.de` endpoint returns HTTP 406 for GET requests without a proper User-Agent).

In [ ]:
overpass_query = """
[out:json][timeout:30];
area["name"="Uster"]["boundary"="administrative"]["admin_level"="8"]->.uster;
(
  node["public_transport"="stop_position"](area.uster);
  node["public_transport"="platform"](area.uster);
  node["highway"="bus_stop"](area.uster);
  node["railway"="tram_stop"](area.uster);
  node["railway"="station"](area.uster);
  node["railway"="halt"](area.uster);
);
out body;
"""

print("Fetching public transport stops from Overpass API...")
headers = {"User-Agent": "PopetyUsterPipeline/1.0"}
mirrors = [
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass-api.de/api/interpreter",
]
for url in mirrors:
    r = requests.post(url, data={"data": overpass_query}, headers=headers, timeout=60)
    if r.ok:
        print(f"Connected via {url}")
        break
r.raise_for_status()

elements = r.json().get("elements", [])
print(f"Raw OSM nodes: {len(elements)}")

stop_rows = []
for el in elements:
    tags = el.get("tags", {})
    stop_rows.append({
        "osm_id":   el["id"],
        "name":     tags.get("name"),
        "type":     tags.get("public_transport") or tags.get("highway") or tags.get("railway"),
        "network":  tags.get("network"),
        "operator": tags.get("operator"),
        "ref":      tags.get("ref"),
        "geometry": Point(el["lon"], el["lat"]),
    })

stops = gpd.GeoDataFrame(stop_rows, crs="EPSG:4326")
stops = stops[stops["name"].notna()].reset_index(drop=True)
stops = stops.drop_duplicates(subset=["name", "geometry"]).reset_index(drop=True)

print(f"Stops after deduplication: {len(stops)}")
print(stops["type"].value_counts())

stops.to_file(str(DATA_DIR / "stops.geojson"), driver="GeoJSON")
print(f"Saved {len(stops)} stops to stops.geojson")
stops[["name", "type", "network", "operator"]].head(10)

## 11. Part 3 – Interactive map

Requirements:
- **Plots with an active construction permit** (blue)
- **Plots within a 3-minute walk of a public transport stop** (green)

Walking speed ≈ 5 km/h → 3 min ≈ **250 m**.
The 250 m buffer is computed in Swiss LV95 (EPSG:2056) for metric accuracy, then reprojected to WGS84.

In [ ]:
# ── Load the three datasets ─────────────────────────────────────────────────────────────
plots_map   = gpd.read_file(str(DATA_DIR / "plots.geojson")).to_crs(epsg=4326)
permits_map = gpd.read_file(str(DATA_DIR / "construction_permits.geojson")).to_crs(epsg=4326)
stops_map   = gpd.read_file(str(DATA_DIR / "stops.geojson")).to_crs(epsg=4326)

# ── Plots with active permits ──────────────────────────────────────────────────────────
# Join permit attributes onto plot geometries so hover shows permit details
joined_permits = gpd.sjoin(permits_map, plots_map, how="inner", predicate="within")

permit_cols = ["title", "pub_date", "project_address", "geocode_status"]
plot_permits = (
    joined_permits.groupby("index_right")[permit_cols]
    .agg(lambda x: " | ".join(x.dropna().astype(str).unique()))
    .reset_index()
    .rename(columns={"index_right": "plot_idx",
                     "title": "permit_titles",
                     "pub_date": "permit_dates",
                     "project_address": "permit_addresses",
                     "geocode_status": "geocode_statuses"})
)

permit_plot_idx   = set(joined_permits["index_right"])
plots_with_permit = plots_map.iloc[sorted(permit_plot_idx)].copy().reset_index()
plots_with_permit = plots_with_permit.rename(columns={"index": "plot_idx"})
plots_with_permit = plots_with_permit.merge(plot_permits, on="plot_idx", how="left")
plots_with_permit = gpd.GeoDataFrame(plots_with_permit, geometry="geometry", crs="EPSG:4326")

# ── Plots within 250 m of a transit stop ─────────────────────────────────────────────
stop_buffer = stops_map.to_crs(epsg=2056).copy()
stop_buffer["geometry"] = stop_buffer.buffer(250)
stop_buffer = stop_buffer.to_crs(epsg=4326)

near_stop_idx  = set(
    gpd.sjoin(plots_map, stop_buffer[["geometry"]], how="inner", predicate="intersects").index
)
plots_near_stop = plots_map.loc[sorted(near_stop_idx)].copy()

print(f"Plots total:             {len(plots_map)}")
print(f"Plots with permit:       {len(plots_with_permit)}")
print(f"Plots near transit stop: {len(plots_near_stop)}")
print(f"Transit stops shown:     {len(stops_map)}")

# ── Build Folium map ────────────────────────────────────────────────────────────────
center = [plots_map.geometry.centroid.y.mean(), plots_map.geometry.centroid.x.mean()]
m3 = folium.Map(location=center, zoom_start=14, tiles="CartoDB positron")

# Layer 1: all plots (gray background)
folium.GeoJson(
    plots_map.__geo_interface__,
    name="All plots",
    style_function=lambda _: {"fillColor": "#d3d3d3", "color": "#aaaaaa", "weight": 0.5, "fillOpacity": 0.4},
    tooltip=folium.GeoJsonTooltip(fields=["egris_egrid", "nummer", "flaechenmass"],
                                   aliases=["EGRID", "Parcel Nr.", "Area (m²)"]),
).add_to(m3)

# Layer 2: plots near transit (green)
folium.GeoJson(
    plots_near_stop.__geo_interface__,
    name="Within 3-min walk of transit",
    style_function=lambda _: {"fillColor": "#2ecc71", "color": "#27ae60", "weight": 1, "fillOpacity": 0.5},
    tooltip=folium.GeoJsonTooltip(fields=["egris_egrid", "nummer"], aliases=["EGRID", "Parcel Nr."]),
).add_to(m3)

# Layer 3: plots with permits (blue) — hover shows permit details
folium.GeoJson(
    plots_with_permit.__geo_interface__,
    name="Plots with construction permit",
    style_function=lambda _: {"fillColor": "#2980b9", "color": "#1a5276", "weight": 1.5, "fillOpacity": 0.65},
    tooltip=folium.GeoJsonTooltip(
        fields=["egris_egrid", "nummer", "flaechenmass",
                "permit_titles", "permit_dates", "permit_addresses"],
        aliases=["EGRID", "Parcel Nr.", "Area (m²)",
                 "Project(s)", "Date(s)", "Address(es)"],
        sticky=True,
    ),
).add_to(m3)

# Layer 4: transit stops (red dots)
stop_layer = folium.FeatureGroup(name="Public transport stops")
for _, row in stops_map.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x], radius=6,
        color="#e74c3c", fill=True, fill_color="#e74c3c", fill_opacity=0.85,
        tooltip=f"{row.get('name', '?')} ({row.get('type', '?')})",
    ).add_to(stop_layer)
stop_layer.add_to(m3)

# Layer 5: 250 m buffers (hidden by default)
folium.GeoJson(
    stop_buffer.__geo_interface__,
    name="250 m walk buffer",
    style_function=lambda _: {"fillColor": "#2ecc71", "color": "#27ae60",
                               "weight": 1, "fillOpacity": 0.08, "dashArray": "4"},
    show=False,
).add_to(m3)

folium.LayerControl(collapsed=False).add_to(m3)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_html = """
<div style="
    position: fixed;
    bottom: 30px; left: 30px;
    background-color: white;
    border: 2px solid #ccc;
    border-radius: 8px;
    padding: 12px 16px;
    font-family: Arial, sans-serif;
    font-size: 13px;
    z-index: 1000;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.2);
">
    <b style="font-size:14px;">Legend</b><br><br>
    <span style="background:#2980b9;width:14px;height:14px;display:inline-block;border-radius:2px;margin-right:6px;"></span>
    Plot with construction permit<br><br>
    <span style="background:#2ecc71;width:14px;height:14px;display:inline-block;border-radius:2px;margin-right:6px;"></span>
    Within 3-min walk of transit<br><br>
    <span style="background:#e74c3c;width:14px;height:14px;display:inline-block;border-radius:50%;margin-right:6px;"></span>
    Public transport stop<br><br>
    <span style="background:#d3d3d3;width:14px;height:14px;display:inline-block;border-radius:2px;margin-right:6px;border:1px solid #aaa;"></span>
    Other plots
</div>
"""
m3.get_root().html.add_child(folium.Element(legend_html))

m3.save(str(DATA_DIR / "part3_map.html"))
print("Saved part3_map.html")
m3

## 12. Elasticsearch – Fuzzy Address Search

The geocoding pipeline fails on ~36% of addresses, mostly due to:
- Abbreviations: `Chammerholzstr.` vs `Chammerholzstrasse`
- Typos: `Rehbühlstasse` vs `Rehbühlstrasse` (missing 'r')
- Letter suffixes not yet in swisstopo: `Krämerackerstrasse 11a`

**Elasticsearch** is a search engine that indexes all known Uster addresses and finds the closest match using **fuzzy matching** (Levenshtein edit distance) — tolerating small differences between the queried string and indexed addresses.

### Why Elasticsearch and not just rapidfuzz?

| | rapidfuzz | Elasticsearch |
|---|---|---|
| Runs | Inside Python script | As a standalone server |
| Multiple users simultaneously | ❌ | ✅ |
| REST API built-in | ❌ | ✅ |
| Sub-100ms at scale | Good for ~100k | Millions of docs |

For a production platform like Popety where many users search simultaneously, Elasticsearch is the natural choice.

### Setup
```bash
# Start Elasticsearch via Docker
docker run -d -p 9200:9200 \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  elasticsearch:8.13.0

pip install "elasticsearch>=8,<9" fastapi uvicorn
```

In [ ]:
from elasticsearch import Elasticsearch, helpers
import time, requests as req

ES_HOST    = "http://localhost:9200"
INDEX_NAME = "swiss_addresses_uster"

def get_client():
    """Connect to local Elasticsearch instance."""
    es = Elasticsearch(ES_HOST)
    if not es.ping():
        raise ConnectionError(
            f"Cannot reach Elasticsearch at {ES_HOST}.\n"
            "Start with: docker run -d -p 9200:9200 "
            "-e 'discovery.type=single-node' "
            "-e 'xpack.security.enabled=false' elasticsearch:8.13.0"
        )
    print(f"Connected to Elasticsearch at {ES_HOST}")
    return es

es = get_client()
print("Cluster info:", es.info()["version"]["number"])

In [ ]:
def create_index(es):
    """Create address index with fuzzy-friendly mapping.
    
    Key fields:
    - address_text: full-text search with standard analyzer (lowercase, tokenise)
    - address_suggest: completion suggester for autocomplete
    - location: geo_point for future spatial queries
    """
    if es.indices.exists(index=INDEX_NAME):
        print(f"Index '{INDEX_NAME}' already exists.")
        return
    mapping = {
        "mappings": {"properties": {
            "address_text":    {"type": "text", "analyzer": "standard",
                                "fields": {"keyword": {"type": "keyword"}}},
            "address_suggest": {"type": "completion"},
            "city":            {"type": "keyword"},
            "zip":             {"type": "keyword"},
            "location":        {"type": "geo_point"},
            "source":          {"type": "keyword"},
        }},
        "settings": {"number_of_shards": 1, "number_of_replicas": 0}
    }
    es.indices.create(index=INDEX_NAME, body=mapping)
    print(f"Created index '{INDEX_NAME}'")


def fetch_uster_addresses():
    """Fetch all Uster addresses from swisstopo by querying a-z prefixes.
    
    swisstopo limits results to 50 per query, so we query once per letter
    to get broad coverage of all registered addresses in Uster (PLZ 8610).
    """
    base_url = "https://api3.geo.admin.ch/rest/services/api/SearchServer"
    all_addresses, seen = [], set()
    for prefix in "abcdefghijklmnopqrstuvwxyz":
        try:
            r = req.get(base_url, params={"type": "locations",
                "searchText": f"{prefix} uster 8610", "lang": "de",
                "limit": 50, "sr": "4326"}, timeout=10)
            for res in r.json().get("results", []):
                attrs = res.get("attrs", {})
                detail = attrs.get("detail", "")
                if detail in seen: continue
                if "8610" not in attrs.get("label","") and "uster" not in attrs.get("label","").lower(): continue
                seen.add(detail)
                if attrs.get("lat") and attrs.get("lon"):
                    all_addresses.append({"detail": detail,
                        "lat": float(attrs["lat"]), "lon": float(attrs["lon"])})
            time.sleep(0.2)
        except Exception as e:
            print(f"  Error on '{prefix}': {e}")
    print(f"Fetched {len(all_addresses)} unique addresses")
    return all_addresses


def build_index(es):
    """Fetch addresses from swisstopo and bulk-index into Elasticsearch."""
    create_index(es)
    raw = fetch_uster_addresses()
    def docs():
        for addr in raw:
            text = addr["detail"].replace(",", " ").strip()
            yield {"_index": INDEX_NAME, "_source": {
                "address_text": text,
                "address_suggest": {"input": [text]},
                "city": "Uster", "zip": "8610",
                "location": {"lat": addr["lat"], "lon": addr["lon"]},
                "source": "swisstopo",
            }}
    count, errors = helpers.bulk(es, docs(), stats_only=True)
    es.indices.refresh(index=INDEX_NAME)
    print(f"Indexed {count} addresses ({errors} errors)")

build_index(es)

In [ ]:
def geocode_elasticsearch(query: str, fuzziness: str = "AUTO") -> tuple | None:
    """Fuzzy address search using Elasticsearch.
    
    fuzziness='AUTO' allows:
    - 0 edits for words with 1-2 chars
    - 1 edit  for words with 3-5 chars  
    - 2 edits for words with 6+ chars
    
    Example: 'Rehbühlstasse' (missing r) → finds 'Rehbühlstrasse' ✅
    """
    resp = es.search(index=INDEX_NAME, body={
        "query": {"match": {"address_text": {
            "query": query, "fuzziness": fuzziness, "operator": "and"
        }}},
        "size": 3,
    })
    hits = resp["hits"]["hits"]
    if not hits:
        return None
    best = hits[0]
    loc  = best["_source"]["location"]
    print(f"  Score {best['_score']:.1f}: '{best['_source']['address_text']}'")
    return loc["lat"], loc["lon"]


# ── Test cases ────────────────────────────────────────────────────────────────
test_queries = [
    "Chammerholzstr. 10 Uster",       # abbreviation
    "Rehbühlstasse 30 Uster",         # typo (missing 'r')
    "Seestrasse 26 Uster",            # exact
    "Krämerackerstrasse 11 Uster",    # new building
]

print("── Elasticsearch fuzzy search test ──")
for q in test_queries:
    result = geocode_elasticsearch(q)
    status = f"→ ({result[0]:.5f}, {result[1]:.5f})" if result else "→ no match"
    print(f"'{q}'  {status}")

In [ ]:
# ── Optional: start REST API so other users can search ────────────────────────
# Uncomment and run to start the API on http://localhost:8000
#
# from fastapi import FastAPI
# import uvicorn
#
# app = FastAPI(title="Swiss Address Search – Uster")
#
# @app.get("/search")
# def search(q: str, size: int = 5):
#     """Fuzzy address search. Example: GET /search?q=Seestrasse+26"""
#     resp = es.search(index=INDEX_NAME, body={
#         "query": {"match": {"address_text": {"query": q, "fuzziness": "AUTO"}}},
#         "size": size,
#     })
#     return [{"address": h["_source"]["address_text"],
#              "lat": h["_source"]["location"]["lat"],
#              "lon": h["_source"]["location"]["lon"],
#              "score": h["_score"]} for h in resp["hits"]["hits"]]
#
# @app.get("/suggest")
# def suggest(q: str):
#     """Autocomplete as user types. Example: GET /suggest?q=Seestra"""
#     resp = es.search(index=INDEX_NAME, body={"suggest": {"s": {
#         "prefix": q, "completion": {"field": "address_suggest",
#         "size": 5, "fuzzy": {"fuzziness": 1}}}}})
#     return [o["text"] for o in resp["suggest"]["s"][0]["options"]]
#
# print("API running at http://localhost:8000")
# print("Try: http://localhost:8000/search?q=Seestrasse+26")
# print("Try: http://localhost:8000/suggest?q=Seestra")
# uvicorn.run(app, host="0.0.0.0", port=8000)

print("Uncomment the block above to start the REST API.")
print(f"Index '{INDEX_NAME}' contains {es.count(index=INDEX_NAME)['count']} addresses.")

## Geocoding Strategy – Summary

| | |
|---|---|
| **Primary geocoder** | swisstopo (`api3.geo.admin.ch`) – Swiss federal address register, returns precise building centroids |
| **Fallback** | Nominatim (OpenStreetMap) – global coverage, useful for named buildings; only accepts `type=house/building` |
| **Validation** | Results outside the Uster municipality boundary are rejected |
| **Multi-address handling** | Titles with multiple streets are exploded into one row per address |
| **Unparseable addresses** | Addresses without a house number (field names, area designations) are skipped |
| **Rate limiting** | 1.5 s sleep between requests; results cached in `geocode_cache.json` |
| **Geometry type** | Parcel polygon (spatial join) when available; otherwise excluded from GeoJSON |
| **What did not work** | Nominatim alone: ~20% miss rate on Swiss address variants; strict Uster boundary validation drops edge cases |

## Appendix A: What We Tried – Geocoding Approaches

### Why swisstopo first, Nominatim second

**swisstopo** (`api3.geo.admin.ch`) is the Swiss federal address register. It covers every officially registered address in Switzerland and returns precise building centroids. It was chosen as the primary geocoder because it handles Swiss-specific address formats (including Uster-specific street names, abbreviated suffixes, and local naming conventions) far better than any global service.

**Nominatim** (OpenStreetMap) was added as a fallback because swisstopo occasionally lacks very new buildings (not yet in the federal register) or named landmarks. Nominatim has global coverage and handles named buildings (e.g. *"Feuerwehr Uster"*) well. However, it is rate-limited to 1 req/s and has a higher miss rate on Swiss address variants.

---

### Approaches that worked

| Approach | What it solves |
|---|---|
| **Street-name validation on swisstopo results** | swisstopo sometimes returns a result for a same-numbered address in a different municipality. We check that the queried street name appears in swisstopo's returned `detail` field. |
| **Nominatim type-check** (`type=house/building` only) | Nominatim interpolates house numbers that don't exist in OSM — it estimates a position between two known numbers and returns `type=residential` or `type=road`. We reject these interpolated results and only accept `type=house` or `type=building`. |
| **Uster boundary validation** | All geocoding results are checked against the dissolved Uster municipality polygon. Results outside the boundary are discarded, preventing false positives from other municipalities. |
| **Decimal normalisation** (`87.2 → 87`) | Some permit titles contained decimal house numbers (likely OCR artefacts from the official gazette PDF). These were normalised with a regex before geocoding. |
| **Letter suffix stripping** (`17a → 17`) | New buildings are often assigned a letter suffix while construction is ongoing but before the address is officially registered. Stripping the suffix and retrying catches many of these. |
| **House number decrement** | If an address is not found, we try progressively lower numbers on the same street (preserving odd/even parity: `17 → 15 → 13 → ...`). This works because new buildings often receive a number adjacent to an existing registered one. |
| **Street-only fallback** | As a last resort, we geocode just the street name. The result is flagged as `needs_review:street_only` and shown in orange on the map. |
| **Geocoding cache** (`geocode_cache.json`) | All geocoding results are cached locally. Subsequent pipeline runs only geocode addresses that have not been seen before — avoiding redundant API calls and rate-limit issues. |

---

### Approaches that did not work well

| Approach | Problem |
|---|---|
| **Reverse geocoding for validation** | Idea: after geocoding a point, reverse-geocode it to confirm the address. In practice, Nominatim's reverse geocoding for Swiss addresses frequently returned business names or POIs at the location (e.g. *"Metzgerei Müller"*, *"Raiffeisenbank"*) rather than the street address — not usable for validation. |
| **Nominatim as sole geocoder** | ~20% miss rate on Swiss address variants: abbreviated street suffixes (`Str.` vs `Strasse`), letter suffixes (`1c`, `106a`), and Swiss-specific street types not well-represented in OSM. |
| **swisstopo without street-name validation** | Without the label-check, swisstopo occasionally returned a plausible-looking result for a street with the same name and house number in a neighbouring municipality (e.g. Seestrasse 26 in Küsnacht instead of Uster). The boundary check alone was not sufficient because the wrong municipality's result could be close to the border. |
| **Street probe check** | An earlier version first verified whether the street existed before searching for the house number, using a separate API call. This doubled the number of requests, significantly worsened the rate-limiting situation, and did not improve hit rate — so it was removed. |
| **Short sleep between requests (0.5 s)** | Led to persistent HTTP 429 (Too Many Requests) errors from swisstopo, halting the pipeline mid-run. Increased to 1.5 s and combined with the cache to resolve this. |

## Appendix B: Future Improvements

### Address extraction

| Idea | Description |
|---|---|
| **NER model for address extraction** | The current regex-based `parse_addresses()` works well for the consistent format of amtsblattportal.ch titles, but edge cases still arise. A named-entity recognition (NER) model fine-tuned on Swiss permit titles could learn to extract addresses more robustly — especially for unusual formats, multiple addresses in one sentence, or cantons with different title conventions. Training data could be bootstrapped from the existing geocoded permits. |
| **LLM-based extraction** | A small instruction-tuned language model (e.g. a quantised Llama or Mistral variant) prompted to extract street addresses from free text would handle edge cases more gracefully than regex, at the cost of inference latency. |

### Geocoding

| Idea | Description |
|---|---|
| **Google Maps Geocoding API** | The current pipeline uses open-source geocoders (swisstopo + Nominatim). Google Maps Geocoding API has significantly better coverage and handles abbreviations, typos, and ambiguous inputs more robustly. It is a paid service but would likely improve the geocoding hit rate — worth evaluating on the ~20% of addresses that currently fail or require fallbacks. |
| **Swiss RegBL (Gebäude- und Wohnungsregister)** | Direct access to the federal building and housing register would provide the most complete and up-to-date Swiss address data, including buildings under construction that are not yet in swisstopo. |
| **Fuzzy address matching** | Use a library like `rapidfuzz` to match extracted addresses against a canonical Swiss address list before geocoding — resolving abbreviations (`Str.` → `Strasse`) and minor typos before the API call, reducing missed addresses. |

### Infrastructure

| Idea | Description |
|---|---|
| **Geocoding cache in PostgreSQL/PostGIS** | The current cache is a local JSON file (`geocode_cache.json`). Storing cached results in the PostGIS database would make them queryable, shareable across machines, and persistent across full environment resets. |
| **Scheduled DAG runs** | The Airflow DAG is currently triggered manually. Adding a monthly schedule would automatically pick up new permits published in the official gazette. Only new addresses (not in the cache) would be geocoded. |
| **Historical permit data** | The current pipeline fetches only the 200 most recent published permits. Loading all historical permits (the API supports pagination) would enable time-series analysis — e.g. construction activity trends by neighbourhood, permit approval-to-completion timelines. |
| **Additional permit types** | Currently filtering only `BP-ZH01` (building permits, Zurich). Extending to demolition permits, zoning changes, and permits from other cantons would give a broader picture of construction activity. |

### RegBL / GWR (Gebäude- und Wohnungsregister)

The Swiss Federal Register of Buildings and Dwellings (GWR) is accessible via the swisstopo MapServer API and contains rich building-level data. It is updated within 48 hours of cantonal/municipal entries and includes fields that go far beyond what the Amtsblatt provides:

| Field | Description |
|---|---|
| `egid` | Unique federal building ID |
| `gstat` | Building status — **1001** = approved, **1002** = permitted, **1003** = under construction, **1004** = completed |
| `gbauj` / `gbaum` | Construction year / month |
| `gabbj` | Demolition year |
| `garea` | Footprint area (m²) |
| `gastw` | Number of floors |
| `ganzwhg` | Number of dwellings |
| `egrid` | Parcel ID — links directly to cadastral data |
| `genh1` / `gwaerzh1` | Heating energy source (useful for energy analysis) |

**Two-step lookup via swisstopo:**

```python
import requests

# Step 1: search by address → get featureId
r = requests.get(
    "https://api3.geo.admin.ch/rest/services/api/SearchServer",
    params={"type": "locations", "searchText": "Seestrasse 26 Uster",
            "lang": "de", "limit": 1, "sr": "4326"},
    timeout=10,
)
feature_id = r.json()["results"][0]["attrs"]["featureId"]
# → "102929_0"

# Step 2: fetch full GWR record
r2 = requests.get(
    f"https://api3.geo.admin.ch/rest/services/ech/MapServer/ch.bfs.gebaeude_wohnungs_register/{feature_id}",
    params={"sr": "4326", "geometryFormat": "geojson"},
    timeout=10,
)
props = r2.json()["feature"]["properties"]
print(props["gstat"])   # 1004 = completed
print(props["gbauj"])   # 1890 = built in 1890
print(props["egrid"])   # CH489731067706 → links to cadastral parcel
```

**Why this matters for this pipeline:** Buildings under construction (`gstat=1003`) could be queried directly from the GWR — potentially replacing or cross-validating the Amtsblatt permit pipeline, without needing address parsing or geocoding at all.